In [2]:
import seaborn as sns
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from tabulate import tabulate
import time

In [3]:

df = pd.read_csv(r"C:\Users\Rasulbek907\Desktop\Project_One\Data\Raw_Data\student_productivity_distraction_dataset_20000.csv")

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             20000 non-null  int64  
 1   age                    20000 non-null  int64  
 2   gender                 20000 non-null  object 
 3   study_hours_per_day    20000 non-null  float64
 4   sleep_hours            20000 non-null  float64
 5   phone_usage_hours      20000 non-null  float64
 6   social_media_hours     20000 non-null  float64
 7   youtube_hours          20000 non-null  float64
 8   gaming_hours           20000 non-null  float64
 9   breaks_per_day         20000 non-null  int64  
 10  coffee_intake_mg       20000 non-null  int64  
 11  exercise_minutes       20000 non-null  int64  
 12  assignments_completed  20000 non-null  int64  
 13  attendance_percentage  20000 non-null  float64
 14  stress_level           20000 non-null  int64  
 15  fo

# Hyperparameters

In [5]:
TARGET_COL = "final_grade"   
BATCH_SIZE = 64
EPOCHS = 50
ALPHA = 1e-3

# Drop NaN in target and Inf

In [6]:
df = df.dropna(subset=[TARGET_COL])
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET_COL])

# Encoding

In [7]:
df_encoded = pd.get_dummies(df, drop_first=True)

In [8]:
X = df_encoded.drop(columns=[TARGET_COL]).values
y = df_encoded[TARGET_COL].values

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (16000, 18)
Test shape: (4000, 18)


# Scaling

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# NaN / Inf tozalash

In [12]:
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0)
y_test  = np.nan_to_num(y_test,  nan=0.0, posinf=0.0, neginf=0.0)

# Log transform target

In [13]:
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# Convert to PyTorch tensors

In [14]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_log, dtype=torch.float32).view(-1, 1)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_log, dtype=torch.float32).view(-1, 1)

In [15]:
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# Neural Network model

In [16]:
torch.manual_seed(0)
input_dim = X_train_t.shape[1]

model = nn.Sequential(
    nn.Linear(input_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

# Train Neural Network

In [18]:
import torch.nn as nn
import torch.optim as optim

# Loss function
loss_fn = nn.MSELoss()

# Optimizer
opt = optim.SGD(model.parameters(), lr=ALPHA)

In [19]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        pred = model(bx)
        loss = loss_fn(pred, by)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {total_loss/len(train_loader):.4f}")

Epoch  10 | Train Loss: 0.1814
Epoch  20 | Train Loss: 0.1436
Epoch  30 | Train Loss: 0.1222
Epoch  40 | Train Loss: 0.1084
Epoch  50 | Train Loss: 0.0990


# Predict with Neural Network (SDL)

In [20]:
model.eval()
with torch.no_grad():
    pred_nn_log = model(X_test_t).cpu().numpy()
    pred_nn = np.expm1(pred_nn_log)  
mae_nn = mean_absolute_error(y_test, pred_nn)
r2_nn  = r2_score(y_test, pred_nn)

print(f"SDL: NeuralNet | MAE: {mae_nn:.2f} | R2: {r2_nn:.4f}")

SDL: NeuralNet | MAE: 17.83 | R2: -0.5844


# Classical ML (SML)

In [21]:
def regression_report(name, y_true, y_pred, train_time):
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    print(f"{name:18s} | MAE: {mae:.2f} | R2: {r2:.4f} | time: {train_time:.3f}s")

In [23]:
model.eval()
with torch.no_grad():
    pred_nn_log = model(X_test_t).cpu().numpy()
    pred_nn = np.expm1(pred_nn_log)
    y_test_real = np.expm1(y_test_t.numpy())

In [ ]:
import time
train_time = 0  
regression_report("NeuralNet", y_test_real, pred_nn, train_time)

NeuralNet          | MAE: 17.83 | R2: -0.5844 | time: 0.000s


In [25]:
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
regression_report("SML: LinearReg", y_test, pred_lr, time.time()-t0)


t0 = time.time()
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)
regression_report("SML: Ridge", y_test, pred_ridge, time.time()-t0)


t0 = time.time()
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
regression_report("SML: RandomForest", y_test, pred_rf, time.time()-t0)

SML: LinearReg     | MAE: 14.94 | R2: -0.0013 | time: 0.033s
SML: Ridge         | MAE: 14.94 | R2: -0.0015 | time: 0.006s
SML: RandomForest  | MAE: 15.05 | R2: -0.0162 | time: 16.109s


In [26]:
from tabulate import tabulate
from sklearn.metrics import mean_absolute_error, r2_score

# ========================
# Natijalarni yig'ish
# ========================
results = [
    ["Linear Regression",        "SML", mean_absolute_error(y_test, pred_lr), r2_score(y_test, pred_lr)],
    ["Ridge Regression",         "SML", mean_absolute_error(y_test, pred_ridge), r2_score(y_test, pred_ridge)],
    ["Random Forest",            "SML", mean_absolute_error(y_test, pred_rf), r2_score(y_test, pred_rf)],
    ["Neural Network (PyTorch)", "SDL", mae_nn, r2_nn],
]

# ========================
# Table chiqarish
# ========================
headers = ["Model", "Type", "MAE", "R2"]
print(tabulate(results, headers=headers, tablefmt="grid", floatfmt=".4f"))

+--------------------------+--------+---------+---------+
| Model                    | Type   |     MAE |      R2 |
+==========================+========+=========+=========+
| Linear Regression        | SML    | 14.9385 | -0.0013 |
+--------------------------+--------+---------+---------+
| Ridge Regression         | SML    | 14.9412 | -0.0015 |
+--------------------------+--------+---------+---------+
| Random Forest            | SML    | 15.0491 | -0.0162 |
+--------------------------+--------+---------+---------+
| Neural Network (PyTorch) | SDL    | 17.8276 | -0.5844 |
+--------------------------+--------+---------+---------+


# 📊 Regression Model Comparison – Xulosa

Ushbu tahlilda talabalar datasetidagi `final_grade` yoki `productivity_score`ni bashorat qilish uchun 4 ta model solishtirildi:

| Model                    | Type | MAE     | R²       |
|--------------------------|------|---------|----------|
| Linear Regression        | SML  | 14.9385 | -0.0013  |
| Ridge Regression         | SML  | 14.9412 | -0.0015  |
| Random Forest            | SML  | 15.0491 | -0.0162  |
| Neural Network (PyTorch) | SDL  | 17.8276 | -0.5844 |

---

## 🔹 Xulosa

1. **Lineer modellar (Linear & Ridge)**
   - MAE ~14.9 va R² ~0, ya’ni model targetni **bashorat qilishda deyarli foydali emas**.
   - Datasetdagi feature-lar va target o‘rtasida kuchli lineer bog‘lanish yo‘q.

2. **Random Forest**
   - MAE ~15.0 va R² ~-0.016, ya’ni lineer modellarga nisbatan yomonroq ishlash yoki overfitting mavjud.
   - Non-linear bog‘lanishlar ham yetarli aniqlanmagan.

3. **Neural Network**
   - MAE ~17.8 va R² ~-0.5844, ya’ni eng yomon natija.
   - Bu holat shuni ko‘rsatadiki, target (final_grade/productivity_score) **highly noisy** yoki feature-lar modelga yetarli signal bermayapti.
   - Neural Networkning log transform va batch training qo‘llangani ham foyda bermagan, ehtimol datasetdagi signal juda past.

---

## 🔹 Tavsiyalar

- Hozirgi dataset bilan regression modellari **amalga oshmayapti**, R² negativ, ya’ni model random guessdan ham yomonroq.  
- Keyingi chora-tadbirlar:  
  1. Feature engineering: yangi muhim feature-lar yaratish yoki irrelevant feature-larni olib tashlash.  
  2. Targetni qayta tekshirish: `final_grade` yoki `productivity_score` juda noaniq yoki noisy bo‘lishi mumkin.  
  3. Agar katta dataset mavjud bo‘lsa: **ensemble metodlar yoki advanced neural network**ni sinash.  
  4. Datasetni normalizatsiya va outlierlarni kamaytirish orqali signalni oshirish.

---

⚠️ Hozirgi natijalar shuni ko‘rsatadiki, **dataset modeli bashorat qilish uchun yetarlicha signal bermayapti**.  
